In [1]:
# Measure the time it takes every cell to run
%pip install ipython-autotime
%load_ext autotime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 20.5 MB/s eta 0:00:00
time: 368 µs (started: 2026-03-24 12:18:59 +00:00)


In [2]:
# Set up idc-index, used to make clinical table parameters human-readable
%pip install idc-index
from idc_index import IDCClient
c = IDCClient()
c.fetch_index('clinical_index')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.3/90.3 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 35.4 MB/s eta 0:00:00
time: 23.5 s (started: 2026-03-24 12:18:59 +00:00)


In [3]:
# Set up BigQuery, used to construct patient cohort of interest
import os
my_ProjectID = "nlst-radiomics"
os.environ["GCP_PROJECT_ID"] = my_ProjectID

from google.colab import auth, files
auth.authenticate_user()

from google.cloud import bigquery
bq_client = bigquery.Client(my_ProjectID)

time: 16.8 s (started: 2026-03-24 12:19:23 +00:00)


In [4]:
# Set up pydicom_seg
%pip install SimpleITK
%pip install pydicom==1.4.2
%pip install pydicom_seg

import SimpleITK as sitk
import pydicom
import pydicom_seg

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 MB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 48.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 81.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.3/122.3 kB 10.1 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: jsonschema
    Found existing installation: jsonschema 4.26.0
    Uninstalling jsonschema-4.26.0:
      Successfully uninstalled jsonschema-4.26.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python-headless 4.13.0.

time: 38.1 s (started: 2026-03-24 12:19:40 +00:00)


In [5]:
# First, what are the possible segmentation types?
# Note: I checked that 'AIMI lung and nodule AI segmentation' contains 'Lung' and 'Nodule' segment labels

seg_query = """
SELECT SeriesDescription
FROM `bigquery-public-data.idc_v23.dicom_all`
WHERE
  collection_id = 'nlst'
  AND Modality = 'SEG'
"""

seg_result = bq_client.query(seg_query)
seg_metadata_df = seg_result.result().to_dataframe()

print(f"\n--- Unique Series Descriptions ---")
display(seg_metadata_df['SeriesDescription'].value_counts().head(20))

print(f"\n--- Unique Series Descriptions Containing 'lung and nodule' ---")
lung_nodule_segmentation_types = seg_metadata_df[seg_metadata_df['SeriesDescription'].str.contains('lung and nodule', case=False)]['SeriesDescription'].unique().tolist()
display(lung_nodule_segmentation_types)

KeyboardInterrupt: 

time: 2.59 s (started: 2026-03-24 12:20:18 +00:00)


In [ ]:
# How many CT/SEG pairs with lung and nodule segmentations are there?
# Note: If seg.SeriesDescription is not radiologist corrected, there are 1042 SEG files with lung and nodule segmentations

ct_seg_join_query = f"""
SELECT
  DISTINCT seg.SeriesInstanceUID AS SEG_SeriesInstanceUID,
  ct.PatientID,
  ct.SeriesInstanceUID AS CT_SeriesInstanceUID,
  seg.SeriesDescription AS SEG_SeriesDescription,
  seg.StudyDate,
  CASE
    WHEN FORMAT_DATE('%Y', seg.StudyDate) LIKE '1999%' THEN 'T0'
    WHEN FORMAT_DATE('%Y', seg.StudyDate) LIKE '2000%' THEN 'T1'
    WHEN FORMAT_DATE('%Y', seg.StudyDate) LIKE '2001%' THEN 'T2'
    ELSE 'Other'
  END AS StudyYear
FROM
  `bigquery-public-data.idc_v23.dicom_all` AS ct
JOIN
  `bigquery-public-data.idc_v23.dicom_all` AS seg
ON
  ct.collection_id = seg.collection_id
  AND ct.PatientID = seg.PatientID
WHERE
  ct.collection_id = 'nlst'
  AND ct.Modality = 'CT'
  AND seg.Modality = 'SEG'
  AND seg.SeriesDescription IN ({', '.join([f"'{s}'" for s in lung_nodule_segmentation_types])})
  AND EXISTS (
    SELECT 1
    FROM UNNEST(seg.ReferencedSeriesSequence) AS ref_seq
    WHERE ct.SeriesInstanceUID = ref_seq.SeriesInstanceUID
  )
"""

ct_seg_join_result = bq_client.query(ct_seg_join_query)
ct_seg_join_df = ct_seg_join_result.result().to_dataframe()

print(f"Number of distinct patients with a CT/SEG pair: {len(ct_seg_join_df['PatientID'].unique())}")
print(f"Number of unique CT series: {len(ct_seg_join_df['CT_SeriesInstanceUID'].unique())}")
print(f"Number of CT/SEG pairs: {ct_seg_join_df.shape[0]}")
print(f"Number of distinct study dates: {len(ct_seg_join_df['StudyDate'].unique())}")

In [ ]:
# Two SEG series can correspond to one CT series because of radiologist corrected segmentations. "Keep" the CT/SEG pair with the radiologist corrected segmentation and throw out the other.

priority_segmentation_type = 'AIMI lung and nodule AI segmentation'

# Create a priority column: 0 for non-AI, 1 for AI
ct_seg_join_df['priority'] = ct_seg_join_df['SEG_SeriesDescription'].apply(lambda x: 1 if x == priority_segmentation_type else 0)

# Sort by CT_SeriesInstanceUID and then by priority (0 comes before 1, so non-AI is prioritized)
ct_seg_join_df_filtered = ct_seg_join_df.sort_values(by=['CT_SeriesInstanceUID', 'priority'])

# Drop duplicates, keeping the first (which will be the non-AI one if present, otherwise the first AI one)
ct_seg_join_df_one_to_one = ct_seg_join_df_filtered.drop_duplicates(subset='CT_SeriesInstanceUID', keep='first')

print(f"\nNumber of CT/SEG pairs after filtering: {ct_seg_join_df_one_to_one.shape[0]}")
display(ct_seg_join_df_one_to_one.head())

In [ ]:
# Download CT/SEG pairs for 5 distinct patients using idc_index

# First, ensure the download directory is set on the Colab VM
download_dir_on_colab_vm = "/content/malignant_trial"
if not os.path.exists(download_dir_on_colab_vm):
    os.makedirs(download_dir_on_colab_vm)

# Your existing code to get series IDs
five_malignant_patientIDs = ct_seg_join_df_one_to_one['PatientID'].unique().tolist()[:5]
CT_SEG_seriesIDs = ct_seg_join_df_one_to_one[ct_seg_join_df_one_to_one['PatientID'].isin(five_malignant_patientIDs)][['CT_SeriesInstanceUID', 'SEG_SeriesInstanceUID']].values.flatten().tolist()

print(f"Downloading {len(CT_SEG_seriesIDs)} series for {len(five_malignant_patientIDs)} patients to Colab VM at {download_dir_on_colab_vm}...")
c.download_from_selection(seriesInstanceUID=CT_SEG_seriesIDs, downloadDir=download_dir_on_colab_vm)
print("Download to Colab VM complete.")

In [ ]:
# Compress the downloaded directory into a zip file
#zip_filename = 'malignant_trial_data.zip'
#!zip -r "{zip_filename}" "{download_dir_on_colab_vm}"

# Trigger download to your local machine
#print(f"\nTriggering download of '{zip_filename}' to your local machine...")
#files.download(zip_filename)
#print("Download initiated. Please check your browser's downloads.")

In [ ]:
# Convert CT and SEG series to .nrrd files for each patient and timestep
base_path = '/content/malignant_trial/nlst'

for patient_id in os.listdir(base_path):
    patient_path = os.path.join(base_path, patient_id)
    if not os.path.isdir(patient_path):
        continue

    for timestep_dir in os.listdir(patient_path):
        timestep_path = os.path.join(patient_path, timestep_dir)
        if not os.path.isdir(timestep_path):
            continue

        ct_series_path = None
        seg_series_path = None

        for item in os.listdir(timestep_path):
            full_item_path = os.path.join(timestep_path, item)
            if os.path.isdir(full_item_path):
                if item.startswith('CT_'):
                    ct_series_path = full_item_path
                elif item.startswith('SEG_'):
                    seg_series_path = full_item_path

        if ct_series_path:
            try:
                reader = sitk.ImageSeriesReader()
                dicom_names = reader.GetGDCMSeriesFileNames(ct_series_path)
                if dicom_names:
                    reader.SetFileNames(dicom_names)
                    ct_image = reader.Execute()
                    output_ct_path = os.path.join(timestep_path, 'CT.nrrd')
                    sitk.WriteImage(ct_image, output_ct_path)
                    print(f"Converted CT for patient {patient_id}, timestep {timestep_dir} to {output_ct_path}")
                else:
                    print(f"No DICOM files found in CT series for patient {patient_id}, timestep {timestep_dir}")
            except Exception as e:
                print(f"Failed to convert CT for patient {patient_id}, timestep {timestep_dir}: {e}")
        else:
            print(f"CT series folder not found for patient {patient_id}, timestep {timestep_dir}. Skipping CT and SEG conversion for this timestep.")
            continue # Skip to next timestep if CT is not found

        # Now process SEG series using pydicom_seg
        if seg_series_path: # Ensure CT DICOMs are loaded for context
            try:
                seg_dcm_files = [f for f in os.listdir(seg_series_path) if f.endswith('.dcm')]
                if not seg_dcm_files:
                    print(f"No DICOM SEG files found in SEG series for patient {patient_id}, timestep {timestep_dir}")
                    continue
                seg_dcm_path = os.path.join(seg_series_path, seg_dcm_files[0])
                seg_dcm = pydicom.dcmread(seg_dcm_path)

                reader = pydicom_seg.SegmentReader()
                result = reader.read(seg_dcm)

                for segment_number in result.available_segments:
                    image_data = result.segment_data(segment_number)
                    image = result.segment_image(segment_number)
                    output_seg_path = os.path.join(timestep_path, f'SEG_{segment_number}.nrrd')
                    sitk.WriteImage(image, output_seg_path, True)

                print(f"Converted SEG for patient {patient_id}, timestep {timestep_dir} to {output_seg_path}")
            except Exception as e:
                print(f"Failed to convert SEG for patient {patient_id}, timestep {timestep_dir}: {e}")
        elif seg_series_path: # seg_series_path exists but ct_dcms is empty
            print(f"CT DICOMs not available for SEG conversion for patient {patient_id}, timestep {timestep_dir}")
        else:
            print(f"SEG series folder not found for patient {patient_id}, timestep {timestep_dir}")